In [1]:
from pathlib import Path
import duckdb
import pandas as pd

# Project directories
ROOT = Path.cwd().parent

RAW_DIR = ROOT / "data" / "raw" / "parquet"
PROCESSED_DIR = ROOT / "data" / "processed"

# Create processed directory if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Connect to DuckDB
con = duckdb.connect()

print("Project root:", ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

print("\nNumber of Parquet files:", len(list(RAW_DIR.glob("*.parquet"))))

Project root: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra
Raw data directory: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\raw\parquet
Processed directory: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\processed

Number of Parquet files: 26


In [2]:
parquet_files = sorted(RAW_DIR.glob("*.parquet"))

for file in parquet_files:
    schema = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{file.as_posix()}')"
    ).df()

    print(f"\n{'=' * 60}")
    print("FILE:", file.name)
    print(schema[["column_name", "column_type"]].to_string(index=False))


FILE: 2001.parquet
   column_name column_type
         State     VARCHAR
      District     VARCHAR
        Market     VARCHAR
     Commodity     VARCHAR
       Variety     VARCHAR
         Grade     VARCHAR
  Arrival_Date     VARCHAR
     Min_Price      BIGINT
     Max_Price      DOUBLE
   Modal_Price      BIGINT
Commodity_Code      BIGINT

FILE: 2002.parquet
   column_name column_type
         State     VARCHAR
      District     VARCHAR
        Market     VARCHAR
     Commodity     VARCHAR
       Variety     VARCHAR
         Grade     VARCHAR
  Arrival_Date     VARCHAR
     Min_Price      DOUBLE
     Max_Price      DOUBLE
   Modal_Price      DOUBLE
Commodity_Code      BIGINT

FILE: 2003.parquet
   column_name column_type
         State     VARCHAR
      District     VARCHAR
        Market     VARCHAR
     Commodity     VARCHAR
       Variety     VARCHAR
         Grade     VARCHAR
  Arrival_Date     VARCHAR
     Min_Price      DOUBLE
     Max_Price      DOUBLE
   Modal_Price      DO

In [3]:
# Read all annual Parquet files together
file_paths = [str(file).replace("\\", "/") for file in parquet_files]

file_list = "[" + ", ".join(
    "'" + path + "'" for path in file_paths
) + "]"

con.execute(f"""
    CREATE OR REPLACE VIEW raw_data AS
    SELECT *
    FROM read_parquet({file_list}, union_by_name = true)
""")

# Inspect combined schema
schema = con.execute("""
    DESCRIBE SELECT * FROM raw_data
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,State,VARCHAR,YES,None,None,None
1,District,VARCHAR,YES,None,None,None
2,Market,VARCHAR,YES,None,None,None
3,Commodity,VARCHAR,YES,None,None,None
4,Variety,VARCHAR,YES,None,None,None
5,Grade,VARCHAR,YES,None,None,None
6,Arrival_Date,VARCHAR,YES,None,None,None
7,Min_Price,DOUBLE,YES,None,None,None
8,Max_Price,DOUBLE,YES,None,None,None
9,Modal_Price,DOUBLE,YES,None,None,None


In [4]:
# Check total records and date range
overview = con.execute("""
    SELECT
        COUNT(*) AS Total_Records,
        MIN(TRY_CAST(Arrival_Date AS DATE)) AS First_Date,
        MAX(TRY_CAST(Arrival_Date AS DATE)) AS Last_Date,
        COUNT(DISTINCT Commodity) AS Unique_Commodities,
        COUNT(DISTINCT State) AS Unique_States
    FROM raw_data
""").df()

display(overview)

,Total_Records,First_Date,Last_Date,Unique_Commodities,Unique_States
0,75984017,2001-01-10,2026-04-21,389,34


In [5]:
commodity_audit = con.execute("""
    SELECT
        COUNT(DISTINCT Commodity) AS Original_Unique_Names,

        COUNT(DISTINCT LOWER(TRIM(Commodity)))
            AS Normalized_Unique_Names,

        COUNT(*) FILTER (
            WHERE Commodity IS NULL OR TRIM(Commodity) = ''
        ) AS Missing_Commodity_Records

    FROM raw_data
""").df()

display(commodity_audit)

,Original_Unique_Names,Normalized_Unique_Names,Missing_Commodity_Records
0,389,387,0


In [6]:
commodity_variants = con.execute("""
    SELECT
        LOWER(TRIM(Commodity)) AS Normalized_Name,
        COUNT(DISTINCT Commodity) AS Original_Variants,
        STRING_AGG(DISTINCT Commodity, ' | ') AS Original_Names

    FROM raw_data
    WHERE Commodity IS NOT NULL
    GROUP BY LOWER(TRIM(Commodity))
    HAVING COUNT(DISTINCT Commodity) > 1
    ORDER BY Original_Variants DESC
""").df()

display(commodity_variants)

,Normalized_Name,Original_Variants,Original_Names
0,tulip,2,tulip | Tulip
1,kutki,2,Kutki | kutki


In [7]:

con.execute("""
    CREATE OR REPLACE VIEW selected_raw AS

    SELECT
        *,
        TRIM(Commodity) AS Project_Commodity

    FROM raw_data

    WHERE LOWER(TRIM(Commodity)) IN (
        'onion',
        'potato',
        'tomato',
        'brinjal',
        'green chilli',
        'bhindi (ladies finger)',
        'cauliflower',
        'cabbage',
        'bottle gourd',
        'bitter gourd',
        'pumpkin',
        'carrot',
        'ginger (green)',
        'garlic',
        'capsicum',
        'beetroot',
        'drumstick',
        'amaranthus',
        'cluster beans',
        'onion green',
        'beans',
        'mint (pudina)',
        'spinach',
        'cowpea (veg)',
        'elephant yam (suran)/amorphophallus',
        'elephant yam (suran)',
        'tapioca',
        'pointed gourd (parval)',
        'peas wet',
        'french beans (frasbean)',
        'yam (ratalu)',
        'sweet potato',
        'sponge gourd',
        'methi (leaves)',
        'sweet pumpkin',
        'green peas',
        'indian beans (seam)',
        'tinda',
        'turnip',
        'cowpea (lobia/karamani)',
        'duster beans',
        'surat beans (papadi)',
        'bunch beans',
        'yam',
        'rat tail radish (mogari)',
        'white pumpkin'
    )
""")


In [8]:
selected_summary = con.execute("""
    SELECT
        Project_Commodity,
        COUNT(*) AS Total_Records,
        COUNT(DISTINCT State) AS Unique_States,
        COUNT(DISTINCT Market) AS Unique_Markets,
        MIN(TRY_CAST(Arrival_Date AS DATE)) AS First_Date,
        MAX(TRY_CAST(Arrival_Date AS DATE)) AS Last_Date

    FROM selected_raw

    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(selected_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Total_Records,Unique_States,Unique_Markets,First_Date,Last_Date
0,Amaranthus,249782,15,547,2001-11-22,2025-11-05
1,Beans,234347,29,925,2001-12-20,2026-04-20
2,Beetroot,381459,27,970,2001-11-27,2026-04-20
3,Bhindi (Ladies Finger),1572079,31,2471,2001-04-08,2026-04-20
4,Bitter gourd,1181917,32,2371,2001-04-21,2026-04-20
5,Bottle gourd,1215058,31,2066,2001-04-12,2026-04-20
6,Brinjal,2448410,31,2862,2001-01-27,2026-04-20
7,Bunch Beans,21504,7,157,2002-04-12,2026-04-17
8,Cabbage,1505899,33,2548,2001-11-22,2026-04-20
9,Capsicum,563526,29,1245,2001-11-22,2026-04-20


In [9]:

commodity_check = con.execute("""
    SELECT
        Project_Commodity,
        COUNT(*) AS Total_Records
    FROM selected_raw
    GROUP BY Project_Commodity
    ORDER BY Total_Records DESC
""").df()

display(commodity_check)

print("Total commodities:", len(commodity_check))
print("Total records:", commodity_check["Total_Records"].sum())


,Project_Commodity,Total_Records
0,Potato,3248834
1,Onion,3168490
2,Tomato,2818577
3,Brinjal,2448410
4,Green Chilli,2108000
5,Cauliflower,1701608
6,Bhindi (Ladies Finger),1572079
7,Cabbage,1505899
8,Bottle gourd,1215058
9,Bitter gourd,1181917


Total commodities: 46
Total records: 30337904


In [10]:
date_price_audit = con.execute("""
    SELECT
        Project_Commodity,

        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE TRY_CAST(Arrival_Date AS DATE) IS NULL
        ) AS Invalid_Dates,

        COUNT(*) FILTER (
            WHERE Min_Price IS NULL
        ) AS Missing_Min_Price,

        COUNT(*) FILTER (
            WHERE Max_Price IS NULL
        ) AS Missing_Max_Price,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
        ) AS Missing_Modal_Price,

        COUNT(*) FILTER (
            WHERE Modal_Price < 0
        ) AS Negative_Modal_Price,

        COUNT(*) FILTER (
            WHERE Min_Price > Max_Price
        ) AS Invalid_Price_Ranges

    FROM selected_raw
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(date_price_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Total_Records,Invalid_Dates,Missing_Min_Price,Missing_Max_Price,Missing_Modal_Price,Negative_Modal_Price,Invalid_Price_Ranges
0,Amaranthus,249782,0,0,0,0,0,57
1,Beans,234347,0,0,0,0,0,24
2,Beetroot,381459,0,0,0,0,0,268
3,Bhindi (Ladies Finger),1572079,0,0,0,0,0,717
4,Bitter gourd,1181917,0,0,0,0,0,497
5,Bottle gourd,1215058,0,0,0,0,0,481
6,Brinjal,2448410,0,0,0,0,0,2171
7,Bunch Beans,21504,0,0,0,0,0,0
8,Cabbage,1505899,0,0,0,0,0,694
9,Capsicum,563526,0,0,0,0,0,313


In [11]:
range_audit = con.execute("""
    SELECT
        Project_Commodity,

        COUNT(*) FILTER (
            WHERE Modal_Price < Min_Price
               OR Modal_Price > Max_Price
        ) AS Modal_Outside_Range,

        COUNT(*) FILTER (
            WHERE Min_Price = 0
              AND Max_Price = 0
              AND Modal_Price > 0
        ) AS Zero_Range_With_Positive_Modal,

        COUNT(*) FILTER (
            WHERE Min_Price = 0
               OR Max_Price = 0
               OR Modal_Price = 0
        ) AS Records_With_Any_Zero_Price

    FROM selected_raw
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(range_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Modal_Outside_Range,Zero_Range_With_Positive_Modal,Records_With_Any_Zero_Price
0,Amaranthus,497,440,530
1,Beans,69,45,386
2,Beetroot,392,124,531
3,Bhindi (Ladies Finger),25570,24853,26545
4,Bitter gourd,21412,20915,22208
5,Bottle gourd,24649,24168,25317
6,Brinjal,61190,59019,62252
7,Bunch Beans,5,5,5
8,Cabbage,25306,24612,27320
9,Capsicum,1177,864,1317


In [12]:
remaining_issues = con.execute("""
    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Arrival_Date,
        Min_Price,
        Modal_Price,
        Max_Price

    FROM selected_raw

    WHERE (
        Modal_Price < Min_Price
        OR Modal_Price > Max_Price
    )
    AND NOT (
        Min_Price = 0
        AND Max_Price = 0
        AND Modal_Price > 0
    )

    ORDER BY Project_Commodity, Arrival_Date

    LIMIT 30
""").df()

display(remaining_issues)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,State,District,Market,Variety,Arrival_Date,Min_Price,Modal_Price,Max_Price
0,Amaranthus,Gujarat,Ahmedabad,Ahmedabad (Rajnagar sub yard),Other,2009-03-24,350.0,500.0,0.0
1,Amaranthus,Kerala,Malappuram,Manjeri,Other,2009-09-30,900.0,900.0,0.0
2,Amaranthus,Kerala,Kasargod,Neeleswaram,Other,2012-11-08,3000.0,3000.0,0.0
3,Amaranthus,Kerala,Kozhikode (Calicut),Perambra,Other,2013-02-18,1000.0,1200.0,0.0
4,Amaranthus,Kerala,Alappuzha,Harippad,Other,2014-06-28,2000.0,2400.0,0.0
5,Amaranthus,Kerala,Ernakulam,Angamaly,Amaranthus,2015-02-20,1000.0,1000.0,0.0
6,Amaranthus,Kerala,Wayanad,Manathavady,Other,2017-01-25,1800.0,2000.0,0.0
7,Amaranthus,Kerala,Pathanamthitta,Pandalam,Amaranthus,2017-05-19,2000.0,2200.0,0.0
8,Amaranthus,Kerala,Alappuzha,Cherthala,Other,2017-06-02,6500.0,6600.0,0.0
9,Amaranthus,Kerala,Palakad,Koduvayoor,Other,2017-09-28,1800.0,2400.0,0.0


In [13]:
remaining_pattern_audit = con.execute("""
    SELECT
        Project_Commodity,

        COUNT(*) AS Total_Remaining_Issues,

        COUNT(*) FILTER (
            WHERE Max_Price = 0 AND Min_Price > 0
        ) AS Zero_Max_Positive_Min,

        COUNT(*) FILTER (
            WHERE Min_Price = 0 AND Max_Price > 0
        ) AS Zero_Min_Positive_Max,

        COUNT(*) FILTER (
            WHERE Min_Price > Max_Price
        ) AS Min_Greater_Than_Max,

        COUNT(*) FILTER (
            WHERE Min_Price > 0
              AND Max_Price > 0
              AND Modal_Price > 0
              AND (
                  Modal_Price < Min_Price
                  OR Modal_Price > Max_Price
              )
        ) AS Modal_Outside_Nonzero_Range

    FROM selected_raw

    WHERE (
        Modal_Price < Min_Price
        OR Modal_Price > Max_Price
    )
    AND NOT (
        Min_Price = 0
        AND Max_Price = 0
        AND Modal_Price > 0
    )

    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(remaining_pattern_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Total_Remaining_Issues,Zero_Max_Positive_Min,Zero_Min_Positive_Max,Min_Greater_Than_Max,Modal_Outside_Nonzero_Range
0,Amaranthus,57,56,0,57,1
1,Beans,24,18,0,24,6
2,Beetroot,268,267,0,268,1
3,Bhindi (Ladies Finger),717,716,0,717,1
4,Bitter gourd,497,497,0,497,0
5,Bottle gourd,481,479,0,481,2
6,Brinjal,2171,2169,0,2171,2
7,Cabbage,694,692,0,694,2
8,Capsicum,313,313,0,313,0
9,Carrot,544,543,0,544,1


In [14]:
nonzero_range_samples = con.execute("""
    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Arrival_Date,
        Min_Price,
        Modal_Price,
        Max_Price

    FROM selected_raw

    WHERE Min_Price > 0
      AND Max_Price > 0
      AND Modal_Price > 0
      AND (
          Modal_Price < Min_Price
          OR Modal_Price > Max_Price
      )

    ORDER BY Project_Commodity, Arrival_Date

    LIMIT 50
""").df()

display(nonzero_range_samples)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,State,District,Market,Variety,Arrival_Date,Min_Price,Modal_Price,Max_Price
0,Amaranthus,Tamil Nadu,Erode,Gobichettipalayam,Amaranthus,2024-07-11,3200.0,3200.0,3000.0
1,Beans,Tamil Nadu,Dharmapuri,Palakode,Beans (Whole),2024-06-18,1000.0,1000.0,120.0
2,Beans,Tamil Nadu,Erode,Sathyamangalam,Beans (Whole),2024-06-19,125.0,125.0,120.0
3,Beans,Tamil Nadu,Thirunelveli,Thenkasi,Beans (Whole),2024-06-20,1300.0,1300.0,150.0
4,Beans,Tamil Nadu,Salem,Attur,Beans (Whole),2024-07-03,80.0,80.0,70.0
5,Beans,Tamil Nadu,Thiruchirappalli,Lalgudi,Beans (Whole),2024-07-10,8000.0,8000.0,6000.0
6,Beans,Tamil Nadu,Thiruchirappalli,Lalgudi,Beans (Whole),2024-07-11,8000.0,8000.0,6000.0
7,Beetroot,Tamil Nadu,Sivaganga,Devakottai,Beetroot,2024-07-07,50.0,50.0,44.0
8,Bhindi (Ladies Finger),Tamil Nadu,Vellore,Gudiyatham,Bhindi,2024-06-18,38.0,38.0,30.0
9,Bottle gourd,Tamil Nadu,Thirunelveli,Thenkasi,Bottle Gourd,2024-06-23,110.0,110.0,12.0


In [15]:
severity_audit = con.execute("""
    SELECT
        Project_Commodity,

        COUNT(*) AS Anomaly_Count,

        ROUND(
            AVG(
                ABS(Modal_Price - Min_Price)
                / NULLIF(Min_Price, 0) * 100
            ), 2
        ) AS Avg_Difference_From_Min_Pct,

        ROUND(
            MAX(
                ABS(Modal_Price - Min_Price)
                / NULLIF(Min_Price, 0) * 100
            ), 2
        ) AS Max_Difference_From_Min_Pct

    FROM selected_raw

    WHERE Min_Price > 0
      AND Max_Price > 0
      AND Modal_Price > 0
      AND (
          Modal_Price < Min_Price
          OR Modal_Price > Max_Price
      )

    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(severity_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Anomaly_Count,Avg_Difference_From_Min_Pct,Max_Difference_From_Min_Pct
0,Amaranthus,1,0.00,0.00
1,Beans,6,0.00,0.00
2,Beetroot,1,0.00,0.00
3,Bhindi (Ladies Finger),1,0.00,0.00
4,Bottle gourd,2,0.00,0.00
5,Brinjal,2,0.00,0.00
6,Cabbage,2,0.00,0.00
7,Carrot,1,0.00,0.00
8,Cauliflower,4,25.29,101.16
9,Cluster beans,1,0.00,0.00


In [16]:
duplicate_audit = con.execute("""
    SELECT
        COUNT(*) AS Rows_In_Duplicate_Groups,
        COUNT(DISTINCT duplicate_key) AS Duplicate_Groups
    FROM (
        SELECT
            HASH(
                Project_Commodity,
                State,
                District,
                Market,
                Variety,
                Grade,
                TRY_CAST(Arrival_Date AS DATE),
                Min_Price,
                Max_Price,
                Modal_Price,
                Commodity_Code
            ) AS duplicate_key

        FROM selected_raw

        GROUP BY
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            TRY_CAST(Arrival_Date AS DATE),
            Min_Price,
            Max_Price,
            Modal_Price,
            Commodity_Code

        HAVING COUNT(*) > 1
    )
""").df()

display(duplicate_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Rows_In_Duplicate_Groups,Duplicate_Groups
0,1585,1585


In [17]:
duplicate_audit = con.execute("""
    WITH duplicate_groups AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            TRY_CAST(Arrival_Date AS DATE) AS Parsed_Date,
            Min_Price,
            Max_Price,
            Modal_Price,
            Commodity_Code,
            COUNT(*) AS Group_Size

        FROM selected_raw

        GROUP BY ALL

        HAVING COUNT(*) > 1
    )

    SELECT
        COUNT(*) AS Duplicate_Groups,

        SUM(Group_Size) AS Rows_In_Duplicate_Groups,

        SUM(Group_Size - 1) AS Excess_Duplicate_Rows,

        MAX(Group_Size) AS Largest_Group_Size

    FROM duplicate_groups
""").df()

display(duplicate_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Duplicate_Groups,Rows_In_Duplicate_Groups,Excess_Duplicate_Rows,Largest_Group_Size
0,1585,3170.0,1585.0,2


In [18]:
exact_duplicate_audit = con.execute("""
    WITH duplicate_groups AS (
        SELECT
            State,
            District,
            Market,
            Commodity,
            Variety,
            Grade,
            Arrival_Date,
            Min_Price,
            Max_Price,
            Modal_Price,
            Commodity_Code,
            COUNT(*) AS Group_Size

        FROM selected_raw

        GROUP BY ALL

        HAVING COUNT(*) > 1
    )

    SELECT
        COUNT(*) AS Exact_Duplicate_Groups,
        SUM(Group_Size) AS Rows_In_Exact_Duplicate_Groups,
        SUM(Group_Size - 1) AS Excess_Exact_Duplicate_Rows,
        MAX(Group_Size) AS Largest_Group_Size

    FROM duplicate_groups
""").df()

display(exact_duplicate_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Exact_Duplicate_Groups,Rows_In_Exact_Duplicate_Groups,Excess_Exact_Duplicate_Rows,Largest_Group_Size
0,1585,3170.0,1585.0,2


In [19]:
con.execute("""
    CREATE OR REPLACE VIEW processed_data AS

    SELECT DISTINCT
        TRIM(State) AS State,
        TRIM(District) AS District,
        TRIM(Market) AS Market,
        TRIM(Commodity) AS Source_Commodity,

        CASE
            WHEN LOWER(TRIM(Commodity)) = 'soyabean'
            THEN 'Soybean'
            ELSE TRIM(Commodity)
        END AS Project_Commodity,

        TRIM(Variety) AS Variety,
        TRIM(Grade) AS Grade,

        Arrival_Date,
        TRY_CAST(Arrival_Date AS DATE) AS Parsed_Date,

        Min_Price,
        Max_Price,
        Modal_Price,
        Commodity_Code,

        -- Missing and invalid price flags
        Modal_Price IS NULL AS flag_missing_modal,

        Modal_Price < 0 AS flag_negative_modal,

        -- Zero and incomplete price ranges
        (
            Min_Price = 0
            AND Max_Price = 0
            AND Modal_Price > 0
        ) AS flag_zero_range_positive_modal,

        (
            Min_Price > 0
            AND Max_Price = 0
        ) AS flag_zero_max_positive_min,

        (
            Min_Price = 0
            AND Max_Price > 0
        ) AS flag_zero_min_positive_max,

        (
            Min_Price = 0
            OR Max_Price = 0
        ) AS flag_incomplete_range,

        (
            Min_Price = 0
            OR Max_Price = 0
            OR Modal_Price = 0
        ) AS flag_any_price_zero,

        -- Invalid price relationships
        (
            Min_Price > Max_Price
        ) AS flag_min_gt_max,

        (
            Min_Price > 0
            AND Max_Price > 0
            AND Modal_Price > 0
            AND (
                Modal_Price < Min_Price
                OR Modal_Price > Max_Price
            )
        ) AS flag_modal_outside_range,

        -- Basic eligibility for modal-price forecasting
        (
            TRY_CAST(Arrival_Date AS DATE) IS NOT NULL
            AND Modal_Price > 0
        ) AS is_forecast_eligible

    FROM selected_raw
""")

print("Processed view created successfully.")

Processed view created successfully.


In [20]:
validation = con.execute("""
    SELECT
        COUNT(*) AS Processed_Records,

        COUNT(DISTINCT Project_Commodity) AS Commodities,

        MIN(Parsed_Date) AS First_Date,

        MAX(Parsed_Date) AS Last_Date,

        COUNT(*) FILTER (
            WHERE flag_missing_modal
        ) AS Missing_Modal_Records,

        COUNT(*) FILTER (
            WHERE flag_modal_outside_range
        ) AS Modal_Outside_Range,

        COUNT(*) FILTER (
            WHERE is_forecast_eligible
        ) AS Forecast_Eligible_Records

    FROM processed_data
""").df()

display(validation)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Processed_Records,Commodities,First_Date,Last_Date,Missing_Modal_Records,Modal_Outside_Range,Forecast_Eligible_Records
0,30336319,46,2001-01-10,2026-04-20,0,1603,30336319


In [21]:
missing_value_audit = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
        ) AS Null_Modal_Prices,

        COUNT(*) FILTER (
            WHERE isnan(Modal_Price)
        ) AS NaN_Modal_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
               OR isnan(Modal_Price)
        ) AS Total_Missing_Modal_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price > 0
              AND NOT isnan(Modal_Price)
              AND Modal_Price IS NOT NULL
        ) AS Valid_Positive_Modal_Prices

    FROM selected_raw
""").df()

display(missing_value_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Null_Modal_Prices,NaN_Modal_Prices,Total_Missing_Modal_Prices,Valid_Positive_Modal_Prices
0,0,0,0,30337904


In [22]:
modal_check = con.execute("""
    SELECT
        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
        ) AS Null_Prices,

        COUNT(*) FILTER (
            WHERE NOT isfinite(Modal_Price)
        ) AS Non_Finite_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price <= 0
        ) AS Non_Positive_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price > 0
              AND isfinite(Modal_Price)
        ) AS Positive_Finite_Prices

    FROM selected_raw
""").df()

display(modal_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Total_Records,Null_Prices,Non_Finite_Prices,Non_Positive_Prices,Positive_Finite_Prices
0,30337904,0,0,0,30337904


In [23]:
modal_check_by_commodity = con.execute("""
    SELECT
        Project_Commodity,
        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
        ) AS Null_Prices,

        COUNT(*) FILTER (
            WHERE NOT isfinite(Modal_Price)
        ) AS Non_Finite_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price <= 0
        ) AS Non_Positive_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price > 0
              AND isfinite(Modal_Price)
        ) AS Positive_Finite_Prices

    FROM selected_raw
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(modal_check_by_commodity)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Total_Records,Null_Prices,Non_Finite_Prices,Non_Positive_Prices,Positive_Finite_Prices
0,Amaranthus,249782,0,0,0,249782
1,Beans,234347,0,0,0,234347
2,Beetroot,381459,0,0,0,381459
3,Bhindi (Ladies Finger),1572079,0,0,0,1572079
4,Bitter gourd,1181917,0,0,0,1181917
5,Bottle gourd,1215058,0,0,0,1215058
6,Brinjal,2448410,0,0,0,2448410
7,Bunch Beans,21504,0,0,0,21504
8,Cabbage,1505899,0,0,0,1505899
9,Capsicum,563526,0,0,0,563526


In [24]:
con.execute("""
    CREATE OR REPLACE VIEW processed_data AS

    SELECT
        * REPLACE (
            (
                Parsed_Date IS NOT NULL
                AND Modal_Price IS NOT NULL
                AND isfinite(Modal_Price)
                AND Modal_Price > 0
            ) AS is_forecast_eligible
        )

    FROM processed_data
""")

print("Forecast eligibility rule updated.")

Forecast eligibility rule updated.


In [25]:
con.execute("""
    CREATE OR REPLACE VIEW processed_data AS

    SELECT DISTINCT
        TRIM(State) AS State,
        TRIM(District) AS District,
        TRIM(Market) AS Market,
        TRIM(Commodity) AS Source_Commodity,

        CASE
            WHEN LOWER(TRIM(Commodity)) = 'soyabean'
            THEN 'Soybean'
            ELSE TRIM(Commodity)
        END AS Project_Commodity,

        TRIM(Variety) AS Variety,
        TRIM(Grade) AS Grade,

        Arrival_Date,
        TRY_CAST(Arrival_Date AS DATE) AS Parsed_Date,

        Min_Price,
        Max_Price,
        Modal_Price,
        Commodity_Code,

        Modal_Price IS NULL AS flag_missing_modal,
        Modal_Price < 0 AS flag_negative_modal,

        (
            Min_Price = 0 AND Max_Price = 0
            AND Modal_Price > 0
        ) AS flag_zero_range_positive_modal,

        (
            Min_Price > 0 AND Max_Price = 0
        ) AS flag_zero_max_positive_min,

        (
            Min_Price = 0 AND Max_Price > 0
        ) AS flag_zero_min_positive_max,

        (
            Min_Price = 0 OR Max_Price = 0
        ) AS flag_incomplete_range,

        (
            Min_Price = 0 OR Max_Price = 0 OR Modal_Price = 0
        ) AS flag_any_price_zero,

        Min_Price > Max_Price AS flag_min_gt_max,

        (
            Min_Price > 0
            AND Max_Price > 0
            AND Modal_Price > 0
            AND (
                Modal_Price < Min_Price
                OR Modal_Price > Max_Price
            )
        ) AS flag_modal_outside_range,

        (
            TRY_CAST(Arrival_Date AS DATE) IS NOT NULL
            AND Modal_Price IS NOT NULL
            AND isfinite(Modal_Price)
            AND Modal_Price > 0
        ) AS is_forecast_eligible

    FROM selected_raw
""")

print("Processed view successfully rebuilt.")

Processed view successfully rebuilt.


In [26]:
OUTPUT_FILE = PROCESSED_DIR / "vegetable_prices_cleaned.parquet"

con.execute(f"""
    COPY processed_data
    TO '{OUTPUT_FILE.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", OUTPUT_FILE)
print("File exists:", OUTPUT_FILE.exists())
print("File size (MB):", round(OUTPUT_FILE.stat().st_size / (1024**2), 2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\processed\vegetable_prices_cleaned.parquet
File exists: True
File size (MB): 312.53


In [27]:
saved_check = con.execute(f"""
    SELECT
        COUNT(*) AS Total_Records,
        COUNT(DISTINCT Project_Commodity) AS Commodities,
        MIN(Parsed_Date) AS First_Date,
        MAX(Parsed_Date) AS Last_Date,
        COUNT(*) FILTER (
            WHERE is_forecast_eligible
        ) AS Forecast_Eligible_Records
    FROM read_parquet('{OUTPUT_FILE.as_posix()}')
""").df()

display(saved_check)

,Total_Records,Commodities,First_Date,Last_Date,Forecast_Eligible_Records
0,30336319,46,2001-01-10,2026-04-20,30336319


In [28]:
market_summary = con.execute(f"""
    SELECT
        Project_Commodity,
        COUNT(*) AS Records,
        COUNT(DISTINCT State) AS States,
        COUNT(DISTINCT Market) AS Market_Names,
        COUNT(DISTINCT
            State || '|' || District || '|' || Market
        ) AS Market_Identities,
        COUNT(DISTINCT Variety) AS Varieties,
        COUNT(DISTINCT Grade) AS Grades

    FROM read_parquet('{OUTPUT_FILE.as_posix()}')

    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(market_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Records,States,Market_Names,Market_Identities,Varieties,Grades
0,Amaranthus,249782,15,547,547,4,4
1,Beans,234292,29,925,927,5,9
2,Beetroot,381398,27,970,971,3,12
3,Bhindi (Ladies Finger),1571994,31,2471,2493,2,13
4,Bitter gourd,1181866,32,2371,2389,4,13
5,Bottle gourd,1214983,31,2066,2083,3,14
6,Brinjal,2448357,31,2862,2897,11,14
7,Bunch Beans,21504,7,157,157,2,7
8,Cabbage,1505826,33,2548,2572,3,13
9,Capsicum,563498,29,1245,1253,2,12


In [29]:
final_audit = con.execute(f"""
    SELECT
        COUNT(*) AS Total_Records,

        COUNT(DISTINCT Project_Commodity) AS Commodities,

        COUNT(*) FILTER (
            WHERE Parsed_Date IS NULL
        ) AS Invalid_Dates,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
               OR NOT isfinite(Modal_Price)
               OR Modal_Price <= 0
        ) AS Invalid_Modal_Prices,

        COUNT(*) FILTER (
            WHERE flag_min_gt_max
        ) AS Invalid_Price_Ranges,

        COUNT(*) FILTER (
            WHERE flag_modal_outside_range
        ) AS Modal_Outside_Range,

        COUNT(*) FILTER (
            WHERE flag_zero_range_positive_modal
        ) AS Zero_Range_Positive_Modal,

        COUNT(*) FILTER (
            WHERE is_forecast_eligible
        ) AS Forecast_Eligible_Records

    FROM read_parquet('{OUTPUT_FILE.as_posix()}')
""").df()

display(final_audit)

,Total_Records,Commodities,Invalid_Dates,Invalid_Modal_Prices,Invalid_Price_Ranges,Modal_Outside_Range,Zero_Range_Positive_Modal,Forecast_Eligible_Records
0,30336319,46,0,0,18483,1603,646707,30336319
